<a href="https://colab.research.google.com/github/rendzina/BigDataAndVisualisation/blob/main/Colab/BristolCityCouncilPropertyExample_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bristol City Council land and building assets challenge
## Unit 2 problem definition and suggested solutions

This notebook loads Bristol City Council land and building asset data into PySpark on Google Colab, checks that the read looks sensible, then steps through a short set of questions. For each question you will see the **DataFrame API** first and the same idea in **Spark SQL**, so you can compare both styles.

Google Colab notebook

# Useful links
## Data source
*Bristol City Council land and building assets*

https://www.data.gov.uk/dataset/a98345ad-7f4c-4f6e-882b-e631dc1cc046/bristol-city-council-land-and-building-assets

*Bristol dataset*

https://www.bristol.gov.uk/files/documents/7241-land-property-2023/file

## Useful programming references

Keep these open while coding: they cover CSV reads, dates, `StructType`, and examples.

*Various Spark examples*

https://sparkbyexamples.com/pyspark/pyspark-read-csv-file-into-dataframe/

https://sparkbyexamples.com/pyspark/pyspark-sql-date-and-timestamp-functions/

https://spark.apache.org/examples.html

https://github.com/apache/spark/tree/master/examples/src/main/python

*O'Reilly's Learning Spark reference*

https://www.oreilly.com/library/view/learning-spark-2nd/9781492050032/ch04.html

*Recipe objective: explain StructType and StructField in PySpark*

https://www.projectpro.io/recipes/explain-structtype-and-structfield-pyspark-databricks#:~:text=The%20StructField%20in%20PySpark%20represents,the%20name%20of%20the%20StructField

*Spark StructTypes (official API)*

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.types.StructType.html#:~:text=.StructType%5Bsource%5D-,Construct%20a%20StructType%20by%20adding%20new%20elements%20to%20it%2C%20to,)%2C%20metadata(optional)


## Mount Azure Blob Storage

The teaching CSV is held in Azure Blob Storage. Mounting exposes those files under `/content/...` so Spark can open the dataset with a normal file path.

You may be prompted for connection details when you run the mount helper (these are provided in class). If mounting fails, the CSV path used later in the notebook will not work. The following cell installs the small `mount-azure-blob` package that the mount step needs.

In [ ]:
%%capture
%pip install mount-azure-blob==0.0.3


The next cell calls `mount_storage`, which attaches the blob as a folder under `/content` using the `mount_path` name. Change that name only if your teaching setup uses a different mount.

In [ ]:
from mount_azure_blob import mount_storage
mount_storage(mount_path="bdv-2024-05-09t15-59-02-855z", config_file=None)


## Start PySpark (Colab)

Colab already ships with PySpark and a Java runtime, so there is no separate Spark install step: you import and create a `SparkSession`, which is the entry point for reading data and running queries.

The following code uses `local[*]` so Spark uses the available CPU cores on the VM and returns the `spark` object used for CSV reads and for SQL when you use it.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F  # used in DataFrame API examples below

spark = (
    SparkSession.builder.master("local[*]")
    .appName("bristol_city_council_assets")
    .getOrCreate()
)
spark


## Load data

The next cell reads the CSV into a DataFrame `df`. The path matches the default mount location; change it if you store the file elsewhere (for example under `/content`).

### Explicit schema

Stating the schema when you read the file is useful for several reasons:

1. **Correct types:** CSV is text, so without a schema columns often stay as strings, or `inferSchema` may guess incorrectly (numeric IDs as doubles, dates that do not parse). Here we set dates, integers, and floats so later aggregates and plots behave as expected.
2. **Reliable dates:** We use `dateFormat` together with `DateType()` so values such as `2/2/2024` parse consistently.
3. **Performance:** Inferring schema scans the data twice; giving a schema avoids that extra pass on large files.
4. **Column names:** Headers include spaces and punctuation; a fixed schema keeps names aligned with the source.

You can use `inferSchema=True` for a quick exploratory read, but when you know the file layout, an explicit schema is the usual choice for coursework and repeatable pipelines.

In [ ]:
# CSV from mounted blob (change the path if your copy of the file lives elsewhere)
from pyspark.sql.types import StructType, StringType, IntegerType
from pyspark.sql.types import DateType, LongType, FloatType

schema = (
    StructType()
    .add("Organisation Name", StringType(), True)
    .add("Organisation Code", StringType(), True)
    .add("Effective Date", DateType(), True)
    .add("UPRN", StringType(), True)
    .add("Property ID", IntegerType(), True)
    .add("Property Type", StringType(), True)
    .add("Property Name/Address (Where no UPRN)", StringType(), True)
    .add("Property Address Detail", StringType(), True)
    .add("Secondary Address Detail", StringType(), True)
    .add("Street Number", StringType(), True)
    .add("Street", StringType(), True)
    .add("Town / Post Town", StringType(), True)
    .add("Post Code", StringType(), True)
    .add("Ward", StringType(), True)
    .add("Geo X (Easting)", LongType(), True)
    .add("Geo Y (Northing)", LongType(), True)
    .add("Tenure Type", StringType(), True)
    .add("Ground Lease In", StringType(), True)
    .add("Ground Lease Out", StringType(), True)
    .add("Lease In to Council", StringType(), True)
    .add("Lease Out", StringType(), True)
    .add("Licence In to Council", StringType(), True)
    .add("Licence Out", StringType(), True)
    .add("Sub-lease In to Council", StringType(), True)
    .add("Sub-lease Out", StringType(), True)
    .add("Vacant", StringType(), True)
    .add("Asset Type", StringType(), True)
    .add("Building Size - GIA (M2)", FloatType(), True)
    .add("Site Area (Hectares)", FloatType(), True)
    .add("Occupied by Council / Direct Service Property", StringType(), True)
    .add("Purpose / Asset Category", StringType(), True)
)

df = (
    spark.read.format("csv")
    .options(header="True", inferSchema="False", delimiter=",", dateFormat="d/M/yyyy")
    .schema(schema)
    .load(
        "/content/bdv-2024-05-09t15-59-02-855z/HdiSamples/"
        "BristolCityCouncilLandAndBuildingAssets-2024.csv"
    )
)


## Inspect the DataFrame

`printSchema()` lists column types and nullability. `show()` prints a sample of rows so you can spot read problems before you aggregate or plot.

In [ ]:
df.printSchema()
df.show()

from IPython.display import display

display(df)


In [ ]:
# Optional for Spark SQL: register this DataFrame as a temporary view.
# You do not need SQL at all: every exercise below can be done with the
# DataFrame API alone. SQL is equivalent: Spark often compiles both to the same plan.
df.createOrReplaceTempView("BristolCouncilAssets")


# Problem 1
## How many "properties" are Bristol City Council (BCC) responsible for (as owner, user or manager)?

Each row is one property record in this dataset, so the count of rows answers the question (within how the council defines a property in the file).

### Approach A: PySpark DataFrame API (no SQL)

`df.count()` gives the number of rows. You do not need a temporary view for this.

In [ ]:
property_count_df = df.count()
print(f"Property row count (DataFrame API): {property_count_df}")


### Approach B: Spark SQL

The same answer can be written in SQL after the DataFrame is registered as a view. Some people prefer SQL, others prefer chaining methods on `df`; both compile to similar Spark plans.

In [ ]:
query = """
SELECT COUNT(*) AS count
FROM BristolCouncilAssets
"""
spark.sql(query).show()


### Optional: result as a pandas object

If the result is small, you can call `toPandas()` on the Spark DataFrame. Only do this when the result comfortably fits in the notebook driver memory.

In [ ]:
import pandas as pd

count_sql_df = spark.sql(
    "SELECT COUNT(*) AS count FROM BristolCouncilAssets"
)
print(count_sql_df.toPandas())


# Problem 2
## How many unique "property types" are Bristol City Council responsible for?

### Approach A: DataFrame API

Select `Property Type`, drop duplicates, then count. `distinct().count()` gives the number of different values.

In [ ]:
distinct_types = df.select("Property Type").distinct()
distinct_types.orderBy("Property Type").show(truncate=False)
n_unique = distinct_types.count()
print(f"\nUnique property types (DataFrame API): {n_unique}")


### Approach B: Spark SQL

The same listing and counts expressed in SQL.

In [ ]:
query = """
SELECT DISTINCT `Property Type`
FROM BristolCouncilAssets
ORDER BY `Property Type`
"""
spark.sql(query).show()


The previous SQL cell lists every distinct type. The next SQL cell returns only how many distinct types there are.

In [ ]:
query = """
SELECT COUNT(*) AS count
FROM (SELECT DISTINCT `Property Type` FROM BristolCouncilAssets) types
"""
spark.sql(query).show()


# Problem 3
## How many properties are there in each of the property types?

### Approach A: DataFrame API

`groupBy` with `count` is the DataFrame version of `GROUP BY` in SQL.

In [ ]:
(
    df.groupBy("Property Type")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)


### Approach B: Spark SQL

Ordering by count makes the table easy to scan.

In [ ]:
query = """
SELECT `Property Type`, COUNT(*) AS count
FROM BristolCouncilAssets
GROUP BY `Property Type`
ORDER BY count DESC
"""
spark.sql(query).show()


# Problem 4
## Histogram: total site area (hectares) by property type

We aggregate `Site Area (Hectares)` by `Property Type`, then plot the top categories.

### Approach A: DataFrame API, then plot

Aggregate in Spark, then move a small result to pandas for matplotlib.

In [ ]:
import matplotlib.pyplot as plt

propertytypes = (
    df.groupBy("Property Type")
    .agg(F.sum("Site Area (Hectares)").alias("sum"))
    .orderBy(F.desc("sum"))
    .limit(15)
    .toPandas()
)

plt.figure(1).set_figwidth(15)
plt.title("Top land asset holdings by property type (DataFrame API)")
plt.xlabel("Property Type")
plt.xticks(rotation=90)
plt.ylabel("Total area (Ha)")
plt.bar(propertytypes["Property Type"], propertytypes["sum"])
plt.show()


### Approach B: same chart from Spark SQL

With the same expressions, this should match Approach A.

In [ ]:
query = """
SELECT `Property Type`, SUM(`Site Area (Hectares)`) AS sum
FROM BristolCouncilAssets
GROUP BY `Property Type`
ORDER BY SUM(`Site Area (Hectares)`) DESC
LIMIT 15
"""
propertytypes_sql = spark.sql(query).toPandas()

plt.figure(2).set_figwidth(15)
plt.title("Top land asset holdings by property type (Spark SQL)")
plt.xlabel("Property Type")
plt.xticks(rotation=90)
plt.ylabel("Total area (Ha)")
plt.bar(propertytypes_sql["Property Type"], propertytypes_sql["sum"])
plt.show()


### Compare with a naive pandas bar chart on raw rows

The next cells take the 15 largest **single-row** site areas, not the 15 property types with the largest **total** area. That is why the chart can differ from the aggregated plots above: check how the data is grouped.

In [ ]:
import pandas as pd

df_pd = df.toPandas().sort_values(
    by=["Site Area (Hectares)"], ascending=False
).head(15)
print(df_pd[["Property Type", "Site Area (Hectares)"]].head(5))
print(
    "\nCheck that df_pd is a pandas DataFrame: ",
    isinstance(df_pd, pd.DataFrame),
)


In [ ]:
plt.figure().set_figwidth(15)
axes = plt.axes()
plt.title("Top rows by site area (not summed by property type)")
plt.xlabel("Property Type")
plt.ylabel("Site area (Ha): single row values")
axes.xaxis.set_tick_params(rotation=90)
df_pd.plot(
    x="Property Type", y="Site Area (Hectares)", legend=True, kind="bar", ax=axes
)
plt.show()


# Problem 5
## For each property type, how many properties fall under each tenure type?

### Approach A: DataFrame API

In [ ]:
(
    df.groupBy("Property Type", "Tenure Type")
    .count()
    .orderBy("Property Type", "Tenure Type")
    .show(truncate=False)
)


### Approach B: Spark SQL

In [ ]:
query = """
SELECT `Property Type`, `Tenure Type`, COUNT(*) AS count
FROM BristolCouncilAssets
GROUP BY `Tenure Type`, `Property Type`
ORDER BY `Property Type` ASC, `Tenure Type` ASC
"""
spark.sql(query).show()
